# 06 — Augmentation, dropout, and weight decay

**Prerequisites:** notebook 05's CNN and training/validation distinction.
**Goals:** design a controlled experiment, apply augmentation only to training images,
and compare dropout and weight decay using validation curves.
**Time:** 25–40 minutes. All data are local bundled digits; no downloads.

A transformation must preserve the correct label. A horizontal flip can change the meaning
of a digit, so our default uses only a small rotation and translation. Even these can erase
detail at 8×8, so we resize to 16×16 and inspect the results first.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision import transforms as T
from course_utils import get_digits_loaders, fit, evaluate

augmentation = T.RandomAffine(degrees=8, translate=(0.06, 0.06))
plain_loaders = get_digits_loaders(batch_size=64, image_size=16, seed=42)
xb, yb = next(iter(plain_loaders[0]))
fig, axes = plt.subplots(1, 7, figsize=(11, 2))
for i, ax in enumerate(axes):
    picture = xb[0] if i == 0 else augmentation(xb[0])
    ax.imshow(picture[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title("Original" if i == 0 else "Augmented"); ax.axis("off")
plt.show()


## 1. Decide what changes and what stays fixed

We use the same split seed, model initialization seed, batch size, learning rate, and epoch budget.
Four runs separate augmentation from dropout plus weight decay, then combine them.
Dropout and weight decay are grouped in one run for speed; the exercises separate them further.

This is one controlled pilot, not strong statistical evidence. Random transforms consume RNG
draws, so stochastic training trajectories still differ. Repeat with several seeds to measure stability.


In [ ]:
class RegularizedCNN(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 24, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((2, 2)))
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(192, 10))
    def forward(self, x):
        return self.head(self.features(x))

configs = {
    "baseline": dict(augment=False, dropout=0.0, decay=0.0),
    "augmentation": dict(augment=True, dropout=0.0, decay=0.0),
    "dropout_and_decay": dict(augment=False, dropout=0.25, decay=1e-3),
    "combined": dict(augment=True, dropout=0.25, decay=1e-3),
}
pd.DataFrame(configs).T


## 2. Run the comparisons

`get_digits_loaders` always uses the same disjoint examples for a given split seed. Its
`train_transform` runs only on training examples; validation and test have deterministic resizing.
Inspect the helper source in `course_utils.py` if you want to see that separation.

Dropout randomly removes some activations during training and disables itself in evaluation.
AdamW's weight decay gently shrinks weights at each optimizer step. Excessive regularization
can cause underfitting. There is no promise that these settings outperform the baseline.


In [ ]:
epochs = 2 if SMOKE_TEST else 12
runs = {}
for name, cfg in configs.items():
    seed_everything(42)
    loaders = get_digits_loaders(batch_size=64, image_size=16, seed=42,
                                train_transform=augmentation if cfg["augment"] else None)
    candidate = RegularizedCNN(cfg["dropout"]).to(device)
    history = fit(candidate, loaders[0], loaders[1], epochs=epochs, lr=0.003,
                  weight_decay=cfg["decay"], device=device)
    runs[name] = {"model": candidate, "history": pd.DataFrame(history),
                  "val": evaluate(candidate, loaders[1], device=device)}
    print(name, "best-epoch validation accuracy:", f"{runs[name]['val']['accuracy']:.1%}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, run in runs.items():
    h = run["history"]
    axes[0].plot(h.epoch, h.val_loss, label=name)
    axes[1].plot(h.epoch, h.val_acc, label=name)
axes[0].set_title("Validation loss"); axes[1].set_title("Validation accuracy")
for ax in axes:
    ax.set_xlabel("Epoch"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 3. Look for a gap, not only a winning score

Training metrics are measured while the model changes, with dropout and augmentation active.
Validation metrics use a fixed model and clean images, so validation can occasionally look easier.
A large training/validation gap can suggest overfitting but does not prove its cause.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for name in ["baseline", "combined"]:
    h = runs[name]["history"]
    axes[0].plot(h.epoch, h.train_loss, label=f"{name}: train")
    axes[0].plot(h.epoch, h.val_loss, "--", label=f"{name}: val")
    axes[1].plot(h.epoch, h.train_acc - h.val_acc, label=name)
axes[0].set_title("Loss: solid train, dashed validation")
axes[1].set_title("Train accuracy − validation accuracy")
for ax in axes:
    ax.set_xlabel("Epoch"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
summary = pd.DataFrame([
    {"run": name, "val_loss": run["val"]["loss"], "val_accuracy": run["val"]["accuracy"]}
    for name, run in runs.items()
]).sort_values("val_loss")
selected_name = summary.iloc[0]["run"]
print("Choose by validation loss:", selected_name)
summary


## 4. Evaluate the selected model once

We used validation to compare configurations. Now estimate generalization with the held-out
test partition. If you keep adjusting settings after reading the test score, that score becomes
part of the tuning process and stops being an independent final estimate.


In [ ]:
selected_model = runs[selected_name]["model"]
test_result = evaluate(selected_model, plain_loaders[2], device=device)
print(f"Selected: {selected_name} | test accuracy: {test_result['accuracy']:.1%}")
summary.to_csv(ARTIFACTS / "06_regularization_comparison.csv", index=False)


In [ ]:
# Demonstrate train/eval behavior without changing the trained model.
dropout_demo = nn.Dropout(0.5)
ones = torch.ones(12)
dropout_demo.train(); print("Training:", dropout_demo(ones))
dropout_demo.eval(); print("Evaluation:", dropout_demo(ones))


### Exercise: Separate dropout from weight decay

Train a dropout-only model, then modify the starter to train a decay-only model. Use validation loss for comparison.


In [ ]:
RUN_EXTRA_EXPERIMENTS = False
if RUN_EXTRA_EXPERIMENTS:
    seed_everything(42)
    loaders = get_digits_loaders(batch_size=64, image_size=16, seed=42)
    dropout_only = RegularizedCNN(dropout=0.25).to(device)
    extra_history = fit(dropout_only, loaders[0], loaders[1], epochs=epochs,
                        lr=0.003, weight_decay=0.0, device=device)
    print(pd.DataFrame(extra_history).tail())


<details><summary>Sample answer — open after trying</summary>

For decay only, use `RegularizedCNN(dropout=0.0)` with `weight_decay=1e-3`. Reseed, recreate loaders, and keep the epoch budget fixed. This separates the two interventions.

</details>


### Exercise: Challenge a label-preserving assumption

Show a stronger rotation next to the original. Decide whether a person could still assign the original label reliably.


In [ ]:
strong_transform = T.RandomAffine(degrees=50, translate=(0.25, 0.25))
fig, axes = plt.subplots(1, 2, figsize=(5, 2))
axes[0].imshow(xb[0, 0], cmap="gray"); axes[0].set_title(f"Original: {yb[0].item()}")
axes[1].imshow(strong_transform(xb[0])[0], cmap="gray"); axes[1].set_title("Strong transform")
for ax in axes: ax.axis("off")
plt.show()
# TODO: Repeat with several digits before judging the transform.


<details><summary>Sample answer — open after trying</summary>

Large rotations or translations can change digit meaning or remove strokes. Inspect multiple images from every class. Stronger augmentation is not automatically better.

</details>


### Exercise: Plan a repeatable experiment

Create a table of three random seeds. Enable the optional loop to estimate variability using validation accuracy.


In [ ]:
seed_results = []
RUN_SEED_STUDY = False
if RUN_SEED_STUDY:
    for model_seed in [7, 42, 123]:
        seed_everything(model_seed)
        loaders = get_digits_loaders(batch_size=64, image_size=16, seed=42)
        candidate = RegularizedCNN().to(device)
        fit(candidate, loaders[0], loaders[1], epochs=epochs, lr=0.003, device=device)
        seed_results.append({"seed": model_seed,
                             "val_accuracy": evaluate(candidate, loaders[1], device=device)["accuracy"]})
print(pd.DataFrame(seed_results))


<details><summary>Sample answer — open after trying</summary>

Keep the split seed fixed to study model initialization variability. Report the individual scores, mean, and sample standard deviation; three runs are a small pilot. A different study can vary split seeds as well.

</details>


**Checkpoint:** identify leakage from validation augmentation, explain dropout in `eval()` mode,
and describe how to compare two regularizers fairly.

Next: [07 — Transfer learning with ResNet](07_transfer_learning_resnet.ipynb).
References: [transforms](https://docs.pytorch.org/vision/stable/transforms.html),
[Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html),
[AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html).
